# exp2 7B Phase 0 - self-contained (no GitHub token, no Drive mount)

Generated 2026-08-16 from `experiment 2/colab/00_setup_schema_audit.ipynb`.
**Only the clone cell was replaced**; every pre-registered Phase-0 cell below is
byte-for-byte the committed version.

Config actually loaded: `exp2_colab_config_mvp.json` (MVP scope fork registered
2026-08-16 - model stays Qwen2.5-7B, scope is cut on the update axis).

## How to run
1. Runtime -> Change runtime type -> **A100 GPU** -> Save
2. Runtime -> Run all
3. Walk away. Phase 0 on a 7B base model is expected to take well over an hour,
   most of it generation.

This runs Phase 0 only (contract re-verification, token audit, split freeze,
Gate C0 memory calibration, sparse-reward preflight, 2-update smoke). **It does
not start Stage A.**

## Why this exists instead of the normal notebook 00

`00_setup_schema_audit.ipynb` clones the private repo with a PAT from Colab
Secrets. That PAT is **broken** — verified 2026-08-16, the clone fails with
`remote: Write access to repository not granted` / HTTP 403. Colab's own GitHub
integration (OAuth) reads the private repo fine, so this notebook is opened
through that instead and carries its source inline, touching no PAT anywhere.

**The embedded blob is a SNAPSHOT.** If `experiment 2/src/`,
`experiment 2/vendor/`, either config, `requirements.txt`, or
`eaaj-pilot/src/{metrics,callbacks}.py` changes, this notebook is stale.
Regenerate it with `scripts/build_7b_selfcontained.py` and re-commit; do not
hand-edit the blob. Delete this notebook once the PAT is fixed.

## Deviation logged up front
The config registers "L4 first, escalate on Gate C0". This notebook's GPU gate
requires >= 35 GiB and therefore goes straight to A100. Reason: on 2026-08-16 the
0.5B track at this identical group-8 geometry was measured needing ~27 GiB and
OOM'd on a 22 GiB L4. A 7B base cannot fit where 0.5B did not. Trying L4 first
would spend a full model-download cycle to learn something already measured,
which defeats the "cheaper compute-unit draw" rationale L4-first was registered
for.


In [ ]:
#@title 1 GPU gate - refuse to continue on unsuitable hardware
import torch

assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> A100"

name = torch.cuda.get_device_name(0)
cap = torch.cuda.get_device_capability(0)
total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
bf16 = torch.cuda.is_bf16_supported()

print(f"GPU            : {name}")
print(f"compute cap    : {cap[0]}.{cap[1]}")
print(f"total memory   : {total_gb:.1f} GiB")
print(f"bf16 supported : {bf16}")
print(f"torch          : {torch.__version__}")

problems = []
if cap[0] < 8:
    problems.append(
        f"Architecture too old (cap {cap[0]}.{cap[1]}). The recipe uses bfloat16, "
        f"which needs Ampere (8.0) or newer. T4/V100 will not work.")
if not bf16:
    problems.append("This device has no native bf16 support.")
if total_gb < 35:
    problems.append(
        f"Only {total_gb:.1f} GiB of VRAM. Qwen2.5-7B in bf16 is ~15 GiB of weights "
        f"before any group-8 generation state. Measured evidence from 2026-08-16: "
        f"the 0.5B track at this same group-8 geometry already needed ~27 GiB and "
        f"OOM'd on a 22 GiB L4. Use A100.")

if problems:
    print("\nUnsuitable GPU:")
    for p in problems:
        print("  -", p)
    raise SystemExit("GPU gate failed")
print("\nGPU gate passed")


In [ ]:
#@title 2 Unpack embedded source (no GitHub token required)
# This notebook carries the repo files it needs as a gzip+base64 blob instead of
# cloning the private repo. Same trick the v9 4070 probe used on 2026-08-16, for
# the same reason: the private-repo clone needs a PAT, and getting that PAT's
# fine-grained permissions right repeatedly failed (and leaked the token into
# cell output twice before it was sanitized). No token means neither failure mode
# can happen. The layout below reproduces the sibling-directory structure
# src/pipeline.py depends on (EXP2_ROOT/.. must contain eaaj-pilot/src).
import base64, gzip, io, os, sys, json, tarfile
from pathlib import Path

_B64 = """
...
"""

REPO_DIR = "/content/RLVR"
os.makedirs(REPO_DIR, exist_ok=True)
_raw = gzip.decompress(base64.b64decode("".join(_B64.split())))
with tarfile.open(fileobj=io.BytesIO(_raw), mode="r") as _tar:
    _tar.extractall(REPO_DIR)

EXP2_DIR = f"{REPO_DIR}/experiment 2"
os.makedirs(f"{EXP2_DIR}/data", exist_ok=True)
for _p in sorted(Path(REPO_DIR).rglob("*")):
    if _p.is_file():
        print(" ", _p.relative_to(REPO_DIR))
print("\nunpacked into", REPO_DIR)


In [ ]:
#@title 3 Install pinned dependencies (2-4 min)
# Pins come from experiment 2/requirements.txt unchanged. torch is deliberately
# NOT reinstalled -- Colab's preinstalled CUDA build is kept.
!pip install -q -r "/content/RLVR/experiment 2/requirements.txt"
print("\nInstall finished.")


In [ ]:
#@title 4 Environment check - versions must match the pinned manifest
import importlib.metadata as md
import sys
import torch

EXPECTED = {"trl": "1.6.0", "transformers": "5.13.0", "datasets": "5.0.0",
            "accelerate": "1.14.0", "peft": "0.15.2", "bitsandbytes": "0.49.2",
            "pylatexenc": "2.10"}

print("python", ".".join(map(str, sys.version_info[:3])))
print("torch ", torch.__version__, f"(CUDA {torch.version.cuda})  <- Colab preinstalled")
print()
mismatched = []
for pkg, want in EXPECTED.items():
    try:
        got = md.version(pkg)
    except Exception:
        got = "MISSING"
    if got != want:
        mismatched.append(pkg)
    print(f"  {'ok ' if got == want else 'BAD'} {pkg:<14} want {want:<10} got {got}")

if mismatched:
    print("\nVersion mismatch:", ", ".join(mismatched))
    print("TRL version drift can silently change GRPO semantics. Report before trusting results.")
else:
    print("\nAll pinned packages match the manifest.")

# Import the bundled modules now, in-process, so a missing dependency surfaces in
# seconds rather than after the 7B download.
sys.path.insert(0, "/content/RLVR/experiment 2")
import src.guru_data, src.guru_reward, src.pipeline  # noqa: F401
from vendor.reasoning360_reward_score import codeio, naive_dapo  # noqa: F401
print("Import smoke test passed: bundled data/reward/pipeline modules load cleanly.")


In [ ]:
#@title 5 Pre-download the model and dataset (7B is ~15 GB - several minutes)
# Not strictly required (unlike the 4070 track, this code does not force
# local_files_only), but downloading here isolates a network failure from a
# training failure, and gives one clean progress bar instead of a stall in the
# middle of Phase 0.
import json, subprocess, sys, time

CFG = json.load(open("/content/RLVR/experiment 2/exp2_colab_config_mvp.json"))
MODEL_ID = CFG["model_id"]
MODEL_REVISION = CFG["model_revision"]
DS_SOURCE = CFG["dataset"]["source"]
DS_REVISION = CFG["dataset"]["revision"]
print(f"model  : {MODEL_ID} @ {MODEL_REVISION}")
print(f"dataset: {DS_SOURCE} @ {DS_REVISION}")

code = "from transformers import AutoTokenizer, AutoConfig, AutoModelForCausalLM\n"
code += "from huggingface_hub import snapshot_download\n"
code += f"AutoTokenizer.from_pretrained({MODEL_ID!r}, revision={MODEL_REVISION!r})\n"
code += f"AutoConfig.from_pretrained({MODEL_ID!r}, revision={MODEL_REVISION!r})\n"
code += f"snapshot_download({MODEL_ID!r}, revision={MODEL_REVISION!r})\n"
code += (f"snapshot_download({DS_SOURCE!r}, repo_type='dataset', "
         f"revision={DS_REVISION!r})\n")

for attempt in range(1, 6):
    print(f"Attempt {attempt}/5 ...", flush=True)
    r = subprocess.run([sys.executable, "-c", code], capture_output=True, text=True)
    if r.returncode == 0:
        print("Download successful. Model weights and dataset are cached.")
        break
    print("Failed:", (r.stderr or "").strip().splitlines()[-1:] or "(no stderr)")
    if attempt < 5:
        time.sleep(15)
else:
    raise SystemExit("Could not download from Hugging Face after 5 attempts.")


## Phase 0 - pre-registered cells below are unmodified

In [ ]:
import json, os, sys
from pathlib import Path

# Replaces the original notebook's private-repo clone. The source is already on
# disk from cell 2; everything below defines exactly the same globals the rest of
# this notebook expects, so the pre-registered cells that follow are unmodified.
REPO_DIR = "/content/RLVR"
EXP2_DIR = f"{REPO_DIR}/experiment 2"
if EXP2_DIR not in sys.path:
    sys.path.insert(0, EXP2_DIR)  # only this one goes on sys.path - pipeline.py
# reaches eaaj-pilot/src by explicit file path internally, avoiding a
# top-level `src` package-name collision between the two sibling dirs
# (see experiment 2/src/pipeline.py's module docstring).

import src.guru_data as guru_data
import src.guru_reward as guru_reward
import src.pipeline as pipeline

# MVP scope fork - see EXPERIMENT_2_COLAB_MVP_AMENDMENT.md. This is the ONE
# knob: switch it back to 'exp2_colab_config.json' if and only if the Phase-0
# promotion gate passes. Never switch it mid-run.
CONFIG_NAME = 'exp2_colab_config_mvp.json'
CONFIG = json.load(open(f'{EXP2_DIR}/{CONFIG_NAME}'))
DATA_DIR = Path(EXP2_DIR) / 'data'
DATA_DIR.mkdir(parents=True, exist_ok=True)
MODEL_ID, MODEL_REVISION = CONFIG['model_id'], CONFIG['model_revision']
DATASET_REVISION = CONFIG['dataset']['revision']
print('config loaded:', CONFIG['experiment'])
print('status       :', CONFIG['status'])
print('model        :', MODEL_ID, '| variant:', CONFIG.get('model_variant'))
print('stage_a      : max_steps', CONFIG['stage_a']['max_steps'],
      '| checkpoints', CONFIG['stage_a']['checkpoint_steps'],
      '| group', CONFIG['stage_a']['num_generations'])


In [ ]:
%pip install -q -r "/content/RLVR/experiment 2/requirements.txt"

## Step 1-3 — load the confirmed contract, spot-check rows

`load_all_records` renders every prompt through THIS model's chat template and computes token counts with THIS model's tokenizer — the field names and file paths are pinned/confirmed, but the counts below are still real numbers for this model, not copy-pasted from the 0.5B track.

In [ ]:
math_rows, sim_rows = guru_data.load_all_records(
    MODEL_ID, MODEL_REVISION, DATASET_REVISION,
    stage_a_prompt_suffix=None)
print('Math (stage A) rows:', len(math_rows))
print('Simulation/CodeIO (stage B) rows:', len(sim_rows))
print()
print('--- sample Math row ---')
sample = math_rows[0]
print({k: (v[:300] if isinstance(v, str) else v) for k, v in sample.items() if k != 'extra_info'})
print()
print('--- sample Simulation row ---')
sample = sim_rows[0]
print({k: (v[:300] if isinstance(v, str) else v) for k, v in sample.items() if k != 'extra_info'})

**Sanity check before continuing:** do the two counts above look like the confirmed audit's `stage_a_count`/`stage_b_count` (`data/guru_schema_audit.json`), and does each sample row have a real rendered prompt (not an empty string or a raw message-list repr) and a real ground_truth? If not, stop and investigate — do not proceed on a loader that silently produced garbage.

## Step 4 — token-length audit under this model's tokenizer (GATE 0a re-verification)

In [ ]:
audit_a = guru_data.token_stats(math_rows)
audit_b = guru_data.token_stats(sim_rows)
print('stage A (Math):', audit_a)
print('stage B (Simulation):', audit_b)

gate_0a_threshold = CONFIG['gates']['phase0a_stage_b_p95_prompt_tokens_max']
if audit_b['p95'] > gate_0a_threshold:
    raise SystemExit(
        f"GATE 0a STOP: stage-B p95={audit_b['p95']} > {gate_0a_threshold}. "
        'Escalate GPU tier — do not shrink the batch to force a fit.')
print('GATE 0a: PASS (confirmed audit already found stage-B p95 well under 1024; this just re-verifies)')

## Step 5 — freeze splits (train/eval/probe), model- and geometry-specific

In [ ]:
splits = guru_data.build_exp2_splits(
    MODEL_ID, MODEL_REVISION,
    stage_a_token_limit=CONFIG['stage_a']['token_filter_max'],
    stage_b_token_limit=CONFIG['stage_b']['token_filter_max'],
    stage_b_eval_questions=CONFIG['stage_b']['eval_questions'],
    n_probe=CONFIG['measurement']['probe_questions'],
    dataset_revision=DATASET_REVISION, seed=CONFIG['seed'],
    out_name='exp2_colab_splits.json')
print('stage_a_train:', len(splits['stage_a_train_ids']),
      '| stage_b_train:', len(splits['stage_b_train_ids']),
      '| stage_b_eval:', len(splits['stage_b_eval_ids']),
      '| probe:', splits['probe_actual'], '/', splits['probe_requested'])
if 'probe_shortfall_note' in splits:
    print('WARNING:', splits['probe_shortfall_note'])

## Gate C0 — GPU memory calibration at the REAL group-8 geometry

This is the first time group 8 will run to completion anywhere in this project (the WIN4070 track's own group-8 attempt OOM'd before finishing its smoke). Escalate tier if it doesn't fit — do not shrink `num_generations`/batch below the config to force an L4 fit.

In [ ]:
sa = CONFIG['stage_a']
smoke_ds = guru_data.to_hf_dataset(math_rows[:8])

gate_c0 = pipeline.gate_c0_memory_probe(
    MODEL_ID, CONFIG['peft'], smoke_ds, sa['reward_mode'],
    num_generations=sa['num_generations'],
    per_device_batch=sa['per_device_train_batch_size'],
    grad_accum=sa['gradient_accumulation_steps'],
    max_completion_length=sa['max_completion_length'],
    device='cuda', min_headroom_pct=CONFIG['gates']['gate_c0_memory_headroom_min_pct'],
    learning_rate=sa['learning_rate'])
print(gate_c0)
if not gate_c0['gate_pass']:
    print('Gate C0: escalate to A100 (switch the Colab runtime, then re-run this cell). '
          'This was the EXPECTED outcome per the plan (§1, GPU tier row) - not a surprise.')
else:
    print('Gate C0: PASS on current tier.')

## Phase 0 step 7-8 — smoke test + tightened sparse-reward preflight (GATE 0b)

16 frozen Stage-A prompts x 8 generations, 8 frozen Stage-B prompts x 8 generations. STOP unless >=2 groups have variable COMBINED reward on EACH stage; exact-channel variance is tracked and reported separately (`FINDING_GROUP_SIZE_REWARD_VARIANCE.md` — combined variance alone overstates how much of the group-8 gain is real reasoning signal vs. format-shaping noise).

In [ ]:
model, tokenizer = pipeline.build_peft_model(MODEL_ID, CONFIG['peft'], device='cuda')

stage_a_preflight_rows = [
    r for r in math_rows if r['id'] in set(splits['stage_a_train_ids'])
][:CONFIG['gates']['phase0b_stage_a_preflight_prompts']]
stage_b_preflight_rows = [
    r for r in sim_rows if r['id'] in set(splits['stage_b_train_ids'])
][:CONFIG['gates']['phase0b_stage_b_preflight_prompts']]

preflight_a = pipeline.guru_sparse_reward_preflight(
    model, tokenizer, stage_a_preflight_rows, sa['reward_mode'],
    num_generations=sa['num_generations'],
    min_variable_groups=CONFIG['gates']['phase0b_min_variable_groups'])
sb = CONFIG['stage_b']
preflight_b = pipeline.guru_sparse_reward_preflight(
    model, tokenizer, stage_b_preflight_rows, sb['reward_mode'],
    num_generations=sb['num_generations'],
    min_variable_groups=CONFIG['gates']['phase0b_min_variable_groups'])

for name, pf in [('Stage A (Math)', preflight_a), ('Stage B (Simulation)', preflight_b)]:
    print(f"{name}: combined-variable groups {pf['groups_with_combined_variance']}/{pf['n_prompts']}, "
          f"exact-variable groups {pf['groups_with_exact_variance']}/{pf['n_prompts']}, "
          f"has_grpo_signal={pf['has_grpo_signal']}")

if not (preflight_a['has_grpo_signal'] and preflight_b['has_grpo_signal']):
    raise SystemExit(
        'GATE 0b STOP: fewer than the required variable groups on at least one stage. '
        'Preserve this preflight result and ask the team. Do not add extra shaping reward '
        'beyond the registered exact_plus_boxed_format_0.1 mode; if the failure looks like '
        'a format-compliance problem specifically, consider the Instruct fallback (config '
        "'model_variant_contingency') and log the deviation - don't silently switch.")
print('GATE 0b: PASS on both stages')

**Format-following check (Base vs Instruct contingency, plan §1/§8 item 4):** scan `preflight_a['groups'][*]['completion_tails']` above for repeated failure to emit a well-formed `\boxed{}`. If most completions never attempt the format, that is the specific signal the WIN4070 track's switch to Instruct was responding to at 0.5B scale — flag it before spending Phase 1 compute on a base model that can't be scored.

In [ ]:
model, tokenizer = pipeline.build_peft_model(MODEL_ID, CONFIG['peft'], device='cuda')
smoke_a = guru_data.to_hf_dataset(stage_a_preflight_rows[:8])
smoke_b = guru_data.to_hf_dataset(stage_b_preflight_rows[:8])

for label, ds, mode, geom in [
    ('stage A smoke', smoke_a, sa['reward_mode'], sa),
    ('stage B smoke', smoke_b, sb['reward_mode'], sb),
]:
    from trl import GRPOConfig, GRPOTrainer
    cfg = GRPOConfig(
        output_dir=f'/tmp/exp2_smoke_{label.replace(" ", "_")}', seed=42, max_steps=2,
        learning_rate=geom['learning_rate'], per_device_train_batch_size=geom['per_device_train_batch_size'],
        gradient_accumulation_steps=geom['gradient_accumulation_steps'], num_generations=geom['num_generations'],
        beta=geom['beta'], max_completion_length=geom['max_completion_length'],
        bf16=True, optim='paged_adamw_8bit', gradient_checkpointing=True,
        gradient_checkpointing_kwargs={'use_reentrant': False},
        logging_steps=1, save_strategy='no', report_to='none')
    trainer = GRPOTrainer(model=model, args=cfg, train_dataset=ds,
                          reward_funcs=guru_reward.select_reward_fn(mode), processing_class=tokenizer)
    trainer.train()
    print(label, 'completed 2/2 smoke updates OK')

## Commit reminder

Commit `data/exp2_colab_splits.json` with message prefix `exp2-colab:`. Log this phase's wall time, GPU tier, and Colab compute-unit cost in `eaaj-pilot/compute_log.md` before moving to notebook 01.